In [ ]:
import torch
import torch.nn as nn
import math
import os
import glob
from PIL import Image
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from accelerate import Accelerator
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Same module as Train_Latent_Diffusion.ipynb

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens


In [ ]:
# Visualize Mask on Image

def preview_mask_over_image(img_tensor_01, x1, y1, x2, y2, title="Mask preview"):
    """
    img_tensor_01: [1,3,H,W] on CPU, in [0,1]
    draws a semi-transparent red rectangle over the masked region
    """
    import matplotlib.patches as patches
    img = img_tensor_01[0].permute(1,2,0).numpy()
    H, W = img.shape[:2]

    fig, ax = plt.subplots(figsize=(6,6))
    ax.imshow(img)
    rect = patches.Rectangle((x1, y1), (x2-x1), (y2-y1), linewidth=2, edgecolor='r', facecolor='r', alpha=0.25)
    ax.add_patch(rect)
    ax.set_title(title)
    ax.axis("off")
    plt.show()

In [ ]:
# Class for Inpainting

class InpaintEngine:
    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5"):
        self.accelerator = Accelerator(mixed_precision='fp16')
        self.vae = AutoencoderKL.from_pretrained(pretrained_path, subfolder="vae")
        self.unet = UNet2DConditionModel.from_pretrained(pretrained_path, subfolder="unet")
        self.scheduler = DDPMScheduler.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            subfolder="scheduler"
        )

        # Coordinate Encoder
        self.coord_encoder = nn.Sequential(nn.Linear(4, 32), nn.GELU())

        # Conditional Projection Layer

        self.cond_proj = CondEncoder()

        # Prepare components for mixed precision
        components = [self.vae, self.unet, self.coord_encoder, self.cond_proj]
        self.vae, self.unet, self.coord_encoder, self.cond_proj = \
            self.accelerator.prepare(*components)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) +
            list(self.coord_encoder.parameters()) +
            list(self.cond_proj.parameters()),
            lr = 1e-5)

        self.optimizer = self.accelerator.prepare(self.optimizer)

        # Noise Scheduler
        self.noise_scheduler = DDPMScheduler.from_pretrained(
            pretrained_path, subfolder="scheduler")
        self.accelerator.register_for_checkpointing(self.unet, self.coord_encoder, self.cond_proj, self.optimizer)
        # Prepare for inference
        self.vae.requires_grad_(False)
        self.unet.requires_grad_(False)
        self.coord_encoder.requires_grad_(False)
        self.cond_proj.requires_grad_(False)

        # Define transformation
        self.transform = transforms.Compose([
            #transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)
        ])

    def _create_latent_mask(self, bbox, latent_shape):
        b, _, lh, lw = latent_shape
        masks = []
        for coords in bbox:
            # Convert coordinates to latent space
            x1 = coords[0] * lw
            y1 = coords[1] * lh
            x2 = coords[2] * lw
            y2 = coords[3] * lh

            # Create basic mask
            xx, yy = torch.meshgrid(
                torch.arange(lw, device=self.accelerator.device),
                torch.arange(lh, device=self.accelerator.device))
            mask = ((xx >= x1) & (xx <= x2) & (yy >= y1) & (yy <= y2)).float()
            masks.append(mask)
        return torch.stack(masks).unsqueeze(1)

    def generate_image(self, masked_img, bbox, steps, j):
        # Encode masked image to latent space
        masked_latents = self.vae.encode(masked_img).latent_dist.sample()
        masked_latents = masked_latents * self.vae.config.scaling_factor
        coord_features = self.coord_encoder(bbox)
        condition = torch.cat([
            self.cond_proj(masked_latents),
            coord_features.unsqueeze(1).expand(-1, 64, -1)
        ], dim=-1)
        print(torch.mean(condition))

        # Create latent mask
        latent_masks = self._create_latent_mask(bbox, masked_latents.shape)
        #print(masked_latents.shape)

        # Add noise to latents based on the mask
        noise = torch.randn_like(masked_latents)
        #timesteps = torch.randint(0, self.noise_scheduler.config.num_train_timesteps, (masked_latents.shape[0],),
        #                          device=masked_latents.device)
        noisy_latents = self.noise_scheduler.add_noise(masked_latents * latent_masks, noise * latent_masks, torch.tensor(steps))
        noisy_latents = masked_latents * (1 - latent_masks) + noisy_latents * latent_masks
        latent_input = noisy_latents

        self.scheduler.set_timesteps(steps)

        for i, t in enumerate(self.scheduler.timesteps):
            latent_input = latent_input * latent_masks + masked_latents * (1-latent_masks)

            with torch.no_grad():
                noise_pred = self.unet(
                    latent_input, t,
                    encoder_hidden_states=condition
                ).sample

            # 8d. Scheduler Step
            # Compute de-noised sample using scheduler's update rule
            latent_input = self.scheduler.step(
                noise_pred, t, latent_input
            ).prev_sample


        with torch.no_grad():
              # Decode blended latent while maintaining original regions
              blended_latent = masked_latents * (1 - latent_masks) + latent_input * latent_masks
              path = f"{j}_latent.pt" #change to your path
              torch.save(blended_latent.float().cpu(),path)
              generated = self.vae.decode(blended_latent / self.vae.config.scaling_factor).sample
              generated = generated[0].permute(1, 2, 0).cpu().numpy()
              generated = (generated * 0.5 + 0.5).clip(0, 1)
              masked_img1 = masked_img[0].cpu().permute(1, 2, 0).numpy()
              masked_img1 = (masked_img1 * 0.5 + 0.5).clip(0, 1)
              generated = (generated * 255).astype(np.uint8)
              img = Image.fromarray(generated)
              img.save(f"{j}.png")
              mask = np.all(generated > 250, axis=2)
              generated[mask] = [255, 255, 255]

              plt.figure(figsize=(12, 6))
              plt.subplot(121)
              plt.imshow(masked_img1)
              plt.title("Masked Input")
              plt.axis('off')

              plt.subplot(122)
              plt.imshow(generated)
              plt.title("Generated Output")
              plt.axis('off')
              plt.show()

In [ ]:
import os
import glob
import torch
from PIL import Image

# Utility: Apply a mask to an image using bbox
def apply_mask(img_tensor, bbox_norm):
    """
    Apply a rectangular mask to an image tensor using normalized bbox coordinates.

    Args:
        img_tensor (Tensor): [1,3,H,W] image in [-1,1]
        bbox_norm (tuple): (x1_norm, y1_norm, x2_norm, y2_norm) in [0,1]

    Returns:
        masked_img (Tensor): masked image [1,3,H,W]
        (x1, y1, x2, y2): pixel coordinates
    """
    _, _, H, W = img_tensor.shape
    x1_norm, y1_norm, x2_norm, y2_norm = bbox_norm

    x1, x2 = int(x1_norm * W), int(x2_norm * W)
    y1, y2 = int(y1_norm * H), int(y2_norm * H)

    mask = torch.ones_like(img_tensor)
    mask[:, :, y1:y2, x1:x2] = 0
    masked_img = img_tensor * mask
    return masked_img, (x1, y1, x2, y2)

# Preview utility (optional visual overlay of mask)
def preview_mask_over_image(img_tensor, x1, y1, x2, y2, title="Mask Preview"):
    """
    Draw a bounding box on the masked preview image.

    img_tensor: [1,3,H,W] in [0,1]
    """
    import matplotlib.pyplot as plt
    import numpy as np

    img = img_tensor[0].permute(1, 2, 0).cpu().numpy()
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.title(title)
    plt.gca().add_patch(
        plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
            edgecolor='red', facecolor='none', linewidth=2)
    )
    plt.axis("off")
    plt.show()

# single-image inference function

@torch.no_grad()
def infer_on_single_image(image_path, bbox_norm, generator, out_prefix="output"):
    """
    Run the inpainting generator on a single image with a user-defined bbox mask.

    Args:
        image_path (str): path to input image
        bbox_norm (tuple): (x1_norm, y1_norm, x2_norm, y2_norm) in [0,1]
        generator: your inpainting model object (with .transform and .generate_image)
        out_prefix (str): prefix for saving result images

    Returns:
        Saves generated image(s) using generator.generate_image(...)
    """
    print(f"\n=== Running inference on: {image_path} ===")

    # Load image
    img = Image.open(image_path).convert("RGB")
    img_t = generator.transform(img).unsqueeze(0).to(generator.accelerator.device)

    # Apply mask
    masked_img, (x1, y1, x2, y2) = apply_mask(img_t, bbox_norm)

    print(f"Applied mask: pixel coords = ({x1},{y1}) - ({x2},{y2})")

    # Optional: visualize masked image
    preview = ((masked_img.float().cpu()[0] * 0.5 + 0.5).unsqueeze(0)).clamp(0, 1)
    preview_mask_over_image(
        preview, x1, y1, x2, y2,
        title=f"Mask for {os.path.basename(image_path)}"
    )

    # Prepare modules on GPU
    generator.unet.to("cuda:0")
    generator.coord_encoder.to("cuda:0")
    generator.cond_proj.to("cuda:0")

    # Run model (200 steps same as original)
    out_name = f"{out_prefix}_mask"
    generator.generate_image(masked_img, torch.tensor([bbox_norm], dtype=torch.float32, device="cuda:0"),
                             200, j=out_name)

    print(f"Saved inference results with prefix: {out_name}")

In [ ]:
# Example usage for ONE image

# User chooses image
img_path = "example.png"

# User defines any normalized bbox they want: x1, y1, x2, y2
bbox_user = (0.3, 0.3, 0.7, 0.7)

generator = OutpaintGenerator()
generator.accelerator.load_state('latent_diffusion_checkpoint_path')

# Run inference
infer_on_single_image(img_path, bbox_user, generator, out_prefix="test_run")
